# Diffusione e instabilità 2D — Glicerolo + Colloide

Traduzione in Python/Dedalus del modello MATLAB 1D (`diffphi1d.m`),
esteso a 2D con le equazioni di Navier-Stokes in approssimazione di Boussinesq.

**Equazioni:**
- Incomprimibilità: ∇·u = 0
- Quantità di moto: ρ₀(∂u/∂t + u·∇u) = −∇p + η∇²u − ρ'g ê_z
- Glicerolo: ∂c/∂t + u·∇c = ∇·[Ds(c)·∇c]
- Colloide: ∂φ/∂t + u·∇φ = ∇·[Dc·∇φ + (a·Dc/ρ₀)·φ·(ρ₀+2bc)·∇c]
- Densità: ρ' = b·c + (ρL−ρ₀)·φ

**Geometria:** Fourier in x (periodica), Chebyshev in z (pareti no-slip/no-flux)

**Condizioni iniziali:** c = c₀, φ = φ₀ per z < h/2; c = 0, φ = 0 per z > h/2
(stesso MATLAB, profilo smussato con tanh per evitare oscillazioni di Gibbs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dedalus.public as d3
from scipy.ndimage import gaussian_filter
from ipywidgets import interact

## 1. Parametri fisici

In [ ]:
# === GEOMETRIA ===
h   = 0.95e-3     # [m] altezza cella (stesso MATLAB)
Lz  = h
Lx  = 4 * h       # [m] estensione orizzontale periodica

# === PARAMETRI FISICI ===
rho0 = 997.0      # [kg/m³] densità acqua pura
rhoL = 2200.0     # [kg/m³] densità colloide
b    = 230.0      # [kg/m³] coefficiente espansione solutale glicerolo
g    = 9.81       # [m/s²]
kB   = 1.380649e-23  # [J/K]
T    = 298.15     # [K] temperatura
Tc   = T - 273.15 # [°C]
r    = 11e-9      # [m] raggio particella colloide
a    = 125.0      # [-] coefficiente cross-diffusione colloide
D0   = 1.025e-9   # [m²/s] diffusività glicerolo a c=0
eps  = -1.308e-9  # [m²/s] correzione lineare diffusività glicerolo

# === CONCENTRAZIONI INIZIALI ===
c0   = 0.4        # concentrazione glicerolo (semispazio inferiore)
phi0 = 0.01       # frazione volumetrica colloide (semispazio inferiore)

# === VISCOSITÀ alla concentrazione media c0 (costante per Dedalus) ===
chi  = 0.705
csi  = 2.0
gamma_v = 1 - c0 + (chi*csi*c0*(1-c0)) / (chi*c0 + csi*(1-c0))
eta_g = 12100 * np.exp((-1233 + Tc)*Tc / (9900 + 70*Tc))   # [mPa·s]
eta_w = 1.79  * np.exp((-1230 - Tc)*Tc / (36100 + 360*Tc)) # [mPa·s]
eta   = eta_w**gamma_v * eta_g**(1 - gamma_v) * 0.001       # [Pa·s]
nu    = eta / rho0                                           # [m²/s]

# === DIFFUSIVITÀ ===
Ds = D0 + eps * c0                        # [m²/s] glicerolo a c=c0 (costante)
Dc = kB * T / (6 * np.pi * eta * r)      # [m²/s] colloide (Stokes-Einstein)

print(f"Viscosità η  = {eta*1000:.3f} mPa·s")
print(f"ν            = {nu:.3e} m²/s")
print(f"Ds (glicerolo) = {Ds:.3e} m²/s")
print(f"Dc (colloide)  = {Dc:.3e} m²/s")
print(f"Rapporto Ds/Dc = {Ds/Dc:.0f}")
print()
print(f"Scala temporale diffusione glicerolo: h²/Ds = {h**2/Ds:.0f} s")
print(f"Scala temporale diffusione colloide:  h²/Dc = {h**2/Dc:.0f} s")

## 2. Parametri numerici

In [ ]:
Nx = 64          # modi Fourier in x
Nz = 32          # punti Chebyshev in z
dealias = 3/2

dt_sim     = 0.5      # [s] passo temporale iniziale
t_end      = 2000.0   # [s] tempo finale (stesso MATLAB)
save_every = 40       # salva un frame ogni N step

# Numero di Rayleigh critico (pareti no-slip, temperatura uniforme)
Ra_crit = 27 * np.pi**4 / 4
print(f"Ra_crit = {Ra_crit:.2f}")

## 3. Griglia e campi Dedalus

In [ ]:
coords = d3.CartesianCoordinates('x', 'z')
dist   = d3.Distributor(coords, dtype=np.float64)

xbasis = d3.RealFourier(coords['x'], size=Nx, bounds=(0, Lx), dealias=dealias)
zbasis = d3.ChebyshevT(coords['z'], size=Nz, bounds=(0, Lz), dealias=dealias)

x, z = dist.local_grids(xbasis, zbasis)
ex, ez = coords.unit_vector_fields(dist)

# --- Campi principali ---
p   = dist.Field(name='p',   bases=(xbasis, zbasis))  # pressione
c   = dist.Field(name='c',   bases=(xbasis, zbasis))  # glicerolo
phi = dist.Field(name='phi', bases=(xbasis, zbasis))  # colloide
u   = dist.VectorField(coords, name='u', bases=(xbasis, zbasis))  # velocità

# --- Campi tau (per le BC con metodo tau-Chebyshev) ---
tau_p    = dist.Field(name='tau_p')
tau_u1   = dist.VectorField(coords, name='tau_u1',   bases=xbasis)
tau_u2   = dist.VectorField(coords, name='tau_u2',   bases=xbasis)
tau_c1   = dist.Field(name='tau_c1',   bases=xbasis)
tau_c2   = dist.Field(name='tau_c2',   bases=xbasis)
tau_phi1 = dist.Field(name='tau_phi1', bases=xbasis)
tau_phi2 = dist.Field(name='tau_phi2', bases=xbasis)

print("Campi creati")

## 4. Equazioni del problema (IVP)

In [ ]:
# --- Operatore lift e gradienti tau-corretti ---
lift_basis = zbasis.derivative_basis(1)
lift = lambda A: d3.Lift(A, lift_basis, -1)

grad_u   = d3.grad(u)   + ez*lift(tau_u1)
grad_c   = d3.grad(c)   + ez*lift(tau_c1)
grad_phi = d3.grad(phi) + ez*lift(tau_phi1)

# --- Perturbazione di densità ---
# ρ' = b·c + (ρL−ρ₀)·φ
rho_prime = b*c + (rhoL - rho0)*phi

# --- Termine cross-diffusione per φ ---
# In MATLAB: flux_phi = Dc·[∇φ + (a/ρ₀)·φ·(ρ₀+2b·c)·∇c]
# → ∇·flux_phi trattato come termine nonlineare nel RHS
cross_phi = (a * Dc / rho0) * d3.div(phi * (rho0 + 2*b*c) * grad_c)

# --- Problema ---
problem = d3.IVP(
    [p, c, phi, u, tau_p, tau_u1, tau_u2, tau_c1, tau_c2, tau_phi1, tau_phi2],
    namespace=locals()
)

# Incomprimibilità
problem.add_equation("trace(grad_u) + tau_p = 0")

# Quantità di moto (Boussinesq): densità più alta → forza verso il basso
problem.add_equation(
    "dt(u) - nu*div(grad_u) + grad(p)/rho0 + lift(tau_u2) = "
    "- u@grad(u) - (rho_prime/rho0)*g*ez"
)

# Glicerolo: diffusione + trasporto convettivo
# Ds costante (valutato a c=c0); la variazione con c è ~10%, trascurata qui
problem.add_equation(
    "dt(c) - Ds*div(grad_c) + lift(tau_c2) = "
    "- u@grad(c)"
)

# Colloide: diffusione + cross-diffusione + trasporto convettivo
problem.add_equation(
    "dt(phi) - Dc*div(grad_phi) + lift(tau_phi2) = "
    "- u@grad(phi) - cross_phi"
)

# --- Condizioni al contorno ---
# Pareti: no-slip per velocità
problem.add_equation("u(z=0)  = 0")
problem.add_equation("u(z=Lz) = 0")

# Pareti: no-flux per glicerolo (dc/dz = 0)
problem.add_equation("ez@grad_c(z=0)  = 0")
problem.add_equation("ez@grad_c(z=Lz) = 0")

# Pareti: no-flux per colloide (dφ/dz = 0)
problem.add_equation("ez@grad_phi(z=0)  = 0")
problem.add_equation("ez@grad_phi(z=Lz) = 0")

# Gauge pressione
problem.add_equation("integ(p) = 0")

print("Problema definito")

## 5. Solver e condizioni iniziali

In [ ]:
solver = problem.build_solver(d3.RK222)
solver.stop_sim_time = t_end
print("Solver costruito")

In [ ]:
# Profilo iniziale: step smussato con tanh
# c = c0, φ = φ0 per z < h/2; c = 0, φ = 0 per z > h/2
delta_s = h / 30  # larghezza della transizione (evita oscillazioni di Gibbs)

step = 0.5 * (1 - np.tanh((z - Lz/2) / delta_s))

c['g']   = c0   * step
phi['g'] = phi0 * step

# Piccola perturbazione casuale su φ per innescare l'instabilità
noise = dist.Field(name='noise', bases=(xbasis, zbasis))
noise.fill_random('g', seed=42, distribution='normal', scale=1.0)
pert = 1e-3 * phi0 * noise['g'] * (z * (Lz - z)) / (Lz/2)**2
pert = gaussian_filter(pert, sigma=1)
phi['g'] += pert

u['g'] = 0.0  # velocità inizialmente nulla

# Plot profili iniziali
z1d = z[0, :]  # profilo lungo z a x=0
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(z1d * 1e3, c['g'][0, :] / c0)
axes[0].set(xlabel='z [mm]', ylabel='c / c₀', title='Glicerolo — t=0')
axes[0].grid(True)
axes[1].plot(z1d * 1e3, phi['g'][0, :] / phi0)
axes[1].set(xlabel='z [mm]', ylabel='φ / φ₀', title='Colloide — t=0')
axes[1].grid(True)
plt.tight_layout()
plt.show()
print("Condizioni iniziali impostate")

## 6. Integrazione temporale

In [ ]:
# Strutture per salvare i frame
times   = []
c_all   = []
phi_all = []
Ux_all  = []
Uz_all  = []
umax_all = []

while solver.proceed:
    solver.step(dt_sim)

    if solver.iteration % save_every == 0:
        t_now = solver.sim_time
        umax  = np.max(np.abs(u['g']))
        times.append(t_now)
        c_all.append(c['g'].copy())
        phi_all.append(phi['g'].copy())
        Ux_all.append(u['g'][0].copy())
        Uz_all.append(u['g'][1].copy())
        umax_all.append(umax)
        print(f"t = {t_now:7.1f} s  |  u_max = {umax:.3e} m/s")

print("\nSimulazione finita")
print(f"Frame salvati: {len(times)}")

## 7. Post-processing

In [ ]:
times    = np.array(times)
c_all    = np.array(c_all)
phi_all  = np.array(phi_all)
Ux_all   = np.array(Ux_all)
Uz_all   = np.array(Uz_all)
umax_all = np.array(umax_all)

# Griglia per i plot (scala dealias)
x_plot, z_plot = dist.local_grids(xbasis, zbasis, scales=dealias)
X, Z = np.meshgrid(x_plot[:, 0], z_plot[0, :], indexing='ij')

print(f"Shape campi: {c_all.shape}  (frame, Nx_deal, Nz_deal)")

In [ ]:
# --- Confronto con MATLAB 1D: profili medi in x ---
times_1d = [1, 25, 125, 400]  # indici frame da confrontare
z_norm = z_plot[0, :] / h

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i in times_1d:
    if i < len(times):
        c_mean   = np.mean(c_all[i],   axis=0)  # media in x
        phi_mean = np.mean(phi_all[i], axis=0)
        axes[0].plot(z_norm, c_mean   / c0,   label=f't={times[i]:.0f} s')
        axes[1].plot(z_norm, phi_mean / phi0, label=f't={times[i]:.0f} s')

axes[0].set(xlabel='z/h', ylabel='c / c₀', title='Glicerolo (media in x)')
axes[0].legend(); axes[0].grid(True)
axes[1].set(xlabel='z/h', ylabel='φ / φ₀', title='Colloide (media in x)')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- Numero di Rayleigh solutale nel tempo (come in MATLAB Fig.5) ---
Ra_crit = 27 * np.pi**4 / 4

Ra_all = np.zeros(len(times))

for i, t_i in enumerate(times):
    c_mean   = np.mean(c_all[i],   axis=0)  # profilo 1D medio
    phi_mean = np.mean(phi_all[i], axis=0)

    rho_profile = rho0 + b*c_mean + (rhoL - rho0)*phi_mean
    z_1d = z_plot[0, :]

    drhodz = np.gradient(rho_profile, z_1d)
    unstable = drhodz > 0  # regione instabile (densità cresce verso l'alto)

    if np.any(unstable):
        z_u   = z_1d[unstable]
        rho_u = rho_profile[unstable]
        Dz = z_u.max() - z_u.min()
        Dr = rho_u.max() - rho_u.min()
        if Dz > 0 and Dr > 0:
            Ra_all[i] = g * Dr * Dz**3 * (6*np.pi*r) / (kB*T)

plt.figure(figsize=(8, 4))
plt.semilogy(times, Ra_all / Ra_crit, lw=2)
plt.axhline(1, ls='--', color='k', label=f'Ra_crit = {Ra_crit:.1f}')
plt.xlabel('t [s]')
plt.ylabel('Ra_S / Ra_S*')
plt.title(f'Numero di Rayleigh solutale  (c₀={c0}, φ₀={phi0})')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- Velocità massima nel tempo ---
plt.figure(figsize=(8, 4))
plt.semilogy(times, umax_all + 1e-20)
plt.xlabel('t [s]')
plt.ylabel('|u|_max [m/s]')
plt.title('Crescita della convezione')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- Viewer interattivo dei campi 2D ---
def plot_frame(n=0):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    cf0 = axes[0].contourf(X*1e3, Z*1e3, c_all[n],   levels=40, cmap='Blues')
    plt.colorbar(cf0, ax=axes[0], label='c [-]')
    axes[0].set(xlabel='x [mm]', ylabel='z [mm]', title=f'Glicerolo  t={times[n]:.0f} s')

    cf1 = axes[1].contourf(X*1e3, Z*1e3, phi_all[n], levels=40, cmap='Oranges')
    plt.colorbar(cf1, ax=axes[1], label='φ [-]')
    axes[1].set(xlabel='x [mm]', ylabel='z [mm]', title=f'Colloide  t={times[n]:.0f} s')

    speed = np.sqrt(Ux_all[n]**2 + Uz_all[n]**2)
    cf2 = axes[2].contourf(X*1e3, Z*1e3, speed, levels=30, cmap='Greys')
    skip = 6
    axes[2].quiver(
        X[::skip, ::skip]*1e3, Z[::skip, ::skip]*1e3,
        Ux_all[n][::skip, ::skip], Uz_all[n][::skip, ::skip],
        color='blue', pivot='mid', scale=None
    )
    plt.colorbar(cf2, ax=axes[2], label='|u| [m/s]')
    axes[2].set(xlabel='x [mm]', ylabel='z [mm]', title=f'Velocità  t={times[n]:.0f} s')

    plt.tight_layout()
    plt.show()

interact(plot_frame, n=(0, len(times)-1, 1));

## Note sulla fisica del modello

**Cosa succede:** Il glicerolo (Ds ~ 5×10⁻¹⁰ m²/s) diffonde molto più velocemente del colloide (Dc ~ 5×10⁻¹² m²/s). Nella fase iniziale, il glicerolo sale nella metà superiore, creando un profilo di densità non monotono: c'è una regione dove il colloide è ancora concentrato in basso ma il glicerolo è già diffuso via, rendendo quella zona più leggera rispetto al fluido sovrastante (ricco di glicerolo). Questo *inversione di densità* locale porta all'instabilità convettiva.

**Differenza 1D vs 2D:** In 1D il codice MATLAB calcola solo *se* Ra > Ra_crit. In 2D Dedalus, la convezione **si sviluppa spontaneamente** come campo di velocità, permettendo di vedere le celle convettive e la loro retroazione sui profili di concentrazione.

**Semplificazioni rispetto al MATLAB:**
- Viscosità η costante (valutata a c = c₀)
- Ds costante (valutata a c = c₀); la variazione è ~10%
- Stessa fisica di cross-diffusione colloide-glicerolo

**Per variare i parametri:** modificare `c0`, `phi0` in cella 2 e ri-eseguire le celle da 5 in poi.